In [11]:
import pandas as pd
import numpy as np

# ==========================
# Load Dataset
# ==========================
df = pd.read_excel("Dataset_Procurement.xlsx")

# ==========================
# Helper Functions
# ==========================

def min_max_positive(series):
    if series.max() == series.min():
        return pd.Series(100, index=series.index)

    return 100 * (
        (series - series.min()) /
        (series.max() - series.min())
    )


def min_max_negative(series):
    if series.max() == series.min():
        return pd.Series(100, index=series.index)

    return 100 * (
        1 -
        ((series - series.min()) /
         (series.max() - series.min()))
    )

# ==========================
# Delivery Score
# ==========================



days_late_score = 100 * np.exp(-df["Days Late"] / 10)
lead_time_score = 100 * np.exp(-df["Lead Time Days"] / 40)

on_time_score = np.where(
    df["On Time Delivery"] == 1,
    100,
    0
)

df["Delivery Score"] = (
    0.5 * on_time_score +
    0.3 * days_late_score +
    0.2 * lead_time_score
)

# ==========================
# Cost Score
# ==========================

price_score = min_max_negative(df["Unit Price"])

discount_score = min_max_positive(df["Discount Pct"])

df["Cost Score"] = (
    0.7 * price_score +
    0.3 * discount_score
)

# ==========================
# Risk Score
# ==========================

risk_mapping = {
    "Low": 100,
    "Medium": 60,
    "High": 20
}

df["Risk Score"] = df["Supplier Risk"].map(risk_mapping)

# ==========================
# Strategic Score
# ==========================

strategic_score = np.zeros(len(df))

strategic_score += np.where(
    df["Preferred Supplier"].astype(str).str.lower().isin(
        ["yes", "true", "1"]
    ),
    50,
    0
)

strategic_score += np.where(
    df["Single Source Flag"].astype(str).str.lower().isin(
        ["no", "false", "0"]
    ),
    20,
    0
)

tier_mapping = {
    "Tier 1": 30,
    "Tier 2": 20,
    "Tier 3": 10
}

strategic_score += (
    df["Supplier Tier"]
    .map(tier_mapping)
    .fillna(0)
)

df["Strategic Score"] = strategic_score

# ==========================
# Final Transaction Score
# ==========================

df["Transaction Score"] = (
    0.40 * df["Delivery Score"] +
    0.25 * df["Cost Score"] +
    0.20 * df["Risk Score"] +
    0.15 * df["Strategic Score"]
)



# ==========================
# Dynamic Supplier Score
# ==========================

supplier_scores = (
    df.groupby("Supplier ID")
    .agg(
        Supplier_Score=("Transaction Score", "mean"),
        Total_Orders=("Supplier ID", "count"),
        Avg_ESG=("Supplier ESG Score", "mean"),
        Avg_Days_Late=("Days Late", "mean"),
        Avg_Lead_Time=("Lead Time Days", "mean")
    )
    .reset_index()
)

supplier_scores = supplier_scores.sort_values(
    by="Supplier_Score",
    ascending=False
)

print(supplier_scores.head(20))

supplier_scores.to_excel(
    "Dynamic_Supplier_Scores.xlsx",
    index=False
)

   Supplier ID  Supplier_Score  Total_Orders  Avg_ESG  Avg_Days_Late  \
1      SUP-002       63.034829           345     41.4       4.055072   
3      SUP-004       62.991315           342     52.3       3.885965   
5      SUP-006       62.846135           346     77.2       4.592486   
0      SUP-001       62.833256           320     75.2       3.909375   
9      SUP-010       62.130467           350     41.6       5.068571   
4      SUP-005       55.943629           311     80.5       3.459807   
13     SUP-014       55.910438           347     50.9       3.412104   
11     SUP-012       55.420341           337     67.8       3.842730   
6      SUP-007       55.320466           380     89.1       4.573684   
12     SUP-013       47.738714           374     41.5       3.574866   
2      SUP-003       47.708675           358     55.1       4.195531   
14     SUP-015       47.706391           335     75.7       4.038806   
8      SUP-009       47.198498           351     63.2       3.69

In [12]:
import pandas as pd
import numpy as np

# =========================
# LOAD DATA
# =========================
df = pd.read_excel("Dataset_Procurement.xlsx")

# =========================
# 1. TIME DECAY WEIGHTING (VERY IMPORTANT)
# If no date column exists, we simulate recency using row order
# Recent transactions should matter more than old ones.
# =========================
df["Recency_Weight"] = np.linspace(1, 0.5, len(df))

# =========================
# 2. NORMALIZED PENALTY SCORES
# =========================

# Delivery penalty (more delay = worse)

#Why exponential?

#Because: small delays are acceptable, but large delays are heavily punished

df["Delivery Score"] = 100 * np.exp(-df["Days Late"] / 5)

# Lead time penalty
#Longer lead time = worse performance.
#But slower decay (÷40) because:
#lead time changes are less sensitive than delays

df["Lead Time Score"] = 100 * np.exp(-df["Lead Time Days"] / 40)

# On-time bonus
#On time → 100
#Not on time → 50 (not zero, because still some delivery happened)

df["On Time Score"] = np.where(df["On Time Delivery"] == 1, 100, 50)

# =========================
# 3. COST SCORE
# =========================
df["Price Score"] = 100 * np.exp(-df["Unit Price"] / df["Unit Price"].mean())

df["Discount Score"] = 100 * (df["Discount Pct"] / (df["Discount Pct"].max() + 1))

df["Cost Score"] = (
    0.7 * df["Price Score"] +
    0.3 * df["Discount Score"]
)

# =========================
# 4. RISK SCORE
# =========================
risk_map = {
    "Low": 100,
    "Medium": 60,
    "High": 20
}

df["Risk Score"] = df["Supplier Risk"].map(risk_map)

# =========================
# 5. STRATEGIC SCORE
# =========================
df["Strategic Score"] = (
    np.where(df["Preferred Supplier"].astype(str).str.lower().isin(["yes", "1", "true"]), 40, 0) +
    np.where(df["Single Source Flag"].astype(str).str.lower().isin(["no", "0", "false"]), 20, 0) +
    df["Supplier Tier"].map({
        "Tier 1": 40,
        "Tier 2": 25,
        "Tier 3": 10
    }).fillna(0)
)

# =========================
# 6. ESG SCORE (reduced weight impact but still included)
# =========================
df["ESG Score"] = df["Supplier ESG Score"]

# =========================
# 7. FINAL TRANSACTION SCORE
# =========================
df["Transaction Score"] = (
    0.35 * df["Delivery Score"] +
    0.15 * df["Lead Time Score"] +
    0.20 * df["Cost Score"] +
    0.15 * df["Risk Score"] +
    0.10 * df["Strategic Score"] +
    0.05 * df["ESG Score"]
)

# apply recency weighting
df["Transaction Score"] = df["Transaction Score"] * df["Recency_Weight"]

# =========================
# 8. SUPPLIER LEVEL AGGREGATION
# =========================
supplier_scores = df.groupby("Supplier ID").agg(
    Supplier_Score=("Transaction Score", "mean"),
    Total_Orders=("Supplier ID", "count"),
    Avg_ESG=("ESG Score", "mean"),
    Avg_Days_Late=("Days Late", "mean"),
    Avg_Lead_Time=("Lead Time Days", "mean")
).reset_index()

# =========================
# 9. NORMALIZE FINAL SCORE (0–100 SCALE)
# =========================
min_score = supplier_scores["Supplier_Score"].min()
max_score = supplier_scores["Supplier_Score"].max()

supplier_scores["Supplier_Score"] = 100 * (
    (supplier_scores["Supplier_Score"] - min_score) /
    (max_score - min_score)
)

# =========================
# 10. SUPPLIER TIERING (A/B/C CLASSIFICATION)
# =========================
def tier(score):
    if score >= 80:
        return "A (Preferred)"
    elif score >= 60:
        return "B (Approved)"
    else:
        return "C (Risk)"

supplier_scores["Tier"] = supplier_scores["Supplier_Score"].apply(tier)

# =========================
# 11. SORT RESULTS
# =========================
supplier_scores = supplier_scores.sort_values(
    by="Supplier_Score",
    ascending=False
)

# =========================
# 12. OUTPUT
# =========================
print("\n🔥 FINAL SUPPLIER PERFORMANCE SYSTEM")
print(supplier_scores)

supplier_scores.to_excel("Final_Supplier_System.xlsx", index=False)


🔥 FINAL SUPPLIER PERFORMANCE SYSTEM
   Supplier ID  Supplier_Score  Total_Orders  Avg_ESG  Avg_Days_Late  \
3      SUP-004      100.000000           342     52.3       3.885965   
0      SUP-001       97.086376           320     75.2       3.909375   
1      SUP-002       92.948233           345     41.4       4.055072   
5      SUP-006       92.049356           346     77.2       4.592486   
4      SUP-005       88.718543           311     80.5       3.459807   
6      SUP-007       80.919098           380     89.1       4.573684   
9      SUP-010       76.637262           350     41.6       5.068571   
13     SUP-014       75.247658           347     50.9       3.412104   
11     SUP-012       68.027834           337     67.8       3.842730   
14     SUP-015       47.136248           335     75.7       4.038806   
8      SUP-009       46.857255           351     63.2       3.692308   
12     SUP-013       45.001223           374     41.5       3.574866   
2      SUP-003       42.049